# 02 — 特征工程
**任务：多景拼接 → 云掩膜 → 特征矩阵输出**

测试区：悉尼西部农业区（147°E–149°E，34°S–32.5°S）
时间段：2026年4月（选取最低云量月合成）

## 1. 导入包

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path
from pystac_client import Client
import planetary_computer as pc
import stackstac
import rasterio
from rasterio.transform import from_bounds
import xarray as xr

from src.data_utils import load_config, ensure_dir, print_array_info
from src.indices import calculate_all_indices

# 修复中文显示
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print('所有包导入成功')

所有包导入成功


## 2. 加载配置

In [2]:
cfg = load_config('../configs/config.yaml')

bbox    = cfg['region']['bbox_test']
BBOX    = [bbox['lon_min'], bbox['lat_min'], bbox['lon_max'], bbox['lat_max']]
EPSG    = 32755
CLOUD_MAX  = cfg['sentinel2']['cloud_cover_max']
STAC_URL   = cfg['sentinel2']['stac_endpoint']
TIME_RANGE = '2026-04-01/2026-05-01'
BANDS      = ['B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'SCL']  # SCL = 云掩膜层

print(f'研究区 BBOX : {BBOX}')
print(f'时间范围    : {TIME_RANGE}')
print(f'最大云量    : {CLOUD_MAX}%')
print(f'加载波段    : {BANDS}')

研究区 BBOX : [147.0, -34.0, 149.0, -32.5]
时间范围    : 2026-04-01/2026-05-01
最大云量    : 20%
加载波段    : ['B02', 'B03', 'B04', 'B08', 'B8A', 'B11', 'SCL']


## 3. 搜索并获取所有低云量场景

In [20]:
catalog = Client.open(STAC_URL)

search = catalog.search(
    collections=['sentinel-2-l2a'],
    bbox=BBOX,
    datetime=TIME_RANGE,
    query={'eo:cloud_cover': {'lt': CLOUD_MAX}}
)

items = list(search.items())
print(f'找到场景数: {len(items)}')
print(f'\n{"场景ID":<55} {"日期":<12} {"云量":>6}')
print('-' * 76)
for item in items:
    date  = item.datetime.strftime('%Y-%m-%d')
    cloud = item.properties.get('eo:cloud_cover', 999)
    print(f'{item.id:<55} {date:<12} {cloud:>6.1f}%')

找到场景数: 100

场景ID                                                    日期               云量
----------------------------------------------------------------------------
S2C_MSIL2A_20260501T001111_R073_T55HFE_20260501T041116  2026-05-01      0.0%
S2C_MSIL2A_20260501T001111_R073_T55HFD_20260501T041116  2026-05-01      0.0%
S2C_MSIL2A_20260501T001111_R073_T55HFC_20260501T041116  2026-05-01      0.3%
S2C_MSIL2A_20260501T001111_R073_T55HEE_20260501T041116  2026-05-01      0.0%
S2C_MSIL2A_20260501T001111_R073_T55HED_20260501T041116  2026-05-01      0.7%
S2C_MSIL2A_20260501T001111_R073_T55HEC_20260501T041116  2026-05-01      8.9%
S2C_MSIL2A_20260501T001111_R073_T55HDE_20260501T041116  2026-05-01      0.0%
S2C_MSIL2A_20260501T001111_R073_T55HDD_20260501T041116  2026-05-01     17.7%
S2B_MSIL2A_20260429T002049_R116_T55HED_20260429T033051  2026-04-29     18.5%
S2B_MSIL2A_20260429T002049_R116_T55HDE_20260429T033051  2026-04-29     17.7%
S2C_MSIL2A_20260428T000241_R030_T55HFE_20260428T024215  2026-04-2

## 4. 加载多景数据（含 SCL 云掩膜层）

In [21]:
# 只取云量最低的5个日期
top_dates = sorted(dates.items(), 
                   key=lambda x: np.mean([i.properties.get('eo:cloud_cover', 999) 
                                          for i in x[1]]))[:5]

print('选取日期：')
for date, scene_items in top_dates:
    avg_cloud = np.mean([i.properties.get('eo:cloud_cover', 999) for i in scene_items])
    print(f'  {date}: {len(scene_items)}景, 平均云量 {avg_cloud:.1f}%')

选取日期：
  2026-04-14: 4景, 平均云量 0.0%
  2026-04-28: 5景, 平均云量 0.0%
  2026-04-20: 3景, 平均云量 0.0%
  2026-04-24: 4景, 平均云量 0.0%
  2026-04-18: 4景, 平均云量 1.4%


## 5. 云掩膜处理

Sentinel-2 SCL（Scene Classification Layer）值说明：
- 4 = 植被（保留）
- 5 = 裸土（保留）  
- 6 = 水体（保留）
- 2,3,7,8,9,10,11 = 云、云阴影、雪（掩膜去除）

In [22]:
from tqdm import tqdm
import gc
import warnings

daily_ndvi  = []
daily_bands = []
transform_ref = None

test_dates = sorted(fresh_dates.items(), key=lambda x: len(x[1]))[:3]
print('选取日期:')
for date, sitems in test_dates:
    print(f'  {date}: {len(sitems)} 景')

for date, scene_items in tqdm(test_dates, desc='按日期处理'):
    try:
        day_stack = stackstac.stack(
            scene_items,
            assets=['B02','B03','B04','B08','B11'],
            bounds_latlon=BBOX,
            resolution=10,
            dtype='float64',
            rescale=False,
            epsg=EPSG
        )
        day_data = day_stack.compute()

        b04 = day_data.sel(band='B04').values / 10000.0
        b08 = day_data.sel(band='B08').values / 10000.0
        valid = (b04 > 0) & (b08 > 0) & np.isfinite(b04) & np.isfinite(b08)
        b04[~valid] = np.nan
        b08[~valid] = np.nan

        ndvi_day = (b08 - b04) / (b08 + b04 + 1e-8)
        ndvi_day[~valid] = np.nan

        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            ndvi_max = np.nanmax(ndvi_day, axis=0)
            best_t   = np.nanargmax(ndvi_day, axis=0)

        rows, cols = np.indices(best_t.shape)

        b02 = day_data.sel(band='B02').values / 10000.0
        b03 = day_data.sel(band='B03').values / 10000.0
        b11 = day_data.sel(band='B11').values / 10000.0
        for b in [b02, b03, b04, b08, b11]:
            b[~valid] = np.nan

        daily_ndvi.append(ndvi_max)
        daily_bands.append([
            b02[best_t, rows, cols],
            b03[best_t, rows, cols],
            b04[best_t, rows, cols],
            b08[best_t, rows, cols],
            b11[best_t, rows, cols],
        ])

        if transform_ref is None:
            transform_ref = day_stack.transform
            height, width = ndvi_max.shape

        del day_data, day_stack
        gc.collect()
        valid_pct = np.isfinite(ndvi_max).mean() * 100
        print(f'  {date} 完成，有效像素: {valid_pct:.1f}%')

    except Exception as e:
        print(f'  {date} 跳过: {e}')

print(f'\n共处理 {len(daily_ndvi)} 个日期')

选取日期:
  2026-04-26: 1 景
  2026-04-13: 1 景
  2026-04-11: 1 景


按日期处理:  33%|███▎      | 1/3 [01:40<03:20, 100.37s/it]

  2026-04-26 跳过: All-NaN slice encountered


按日期处理:  67%|██████▋   | 2/3 [02:01<00:53, 53.71s/it] 

  2026-04-13 跳过: All-NaN slice encountered


按日期处理: 100%|██████████| 3/3 [03:06<00:00, 62.33s/it]

  2026-04-11 跳过: All-NaN slice encountered

共处理 0 个日期


In [24]:
from tqdm import tqdm
import gc
import warnings

daily_ndvi  = []
daily_bands = []
transform_ref = None

# 选景数最多的3个日期
best_dates = sorted(fresh_dates.items(), key=lambda x: len(x[1]), reverse=True)[:3]
print('选取日期:')
for date, sitems in best_dates:
    print(f'  {date}: {len(sitems)} 景')

for date, scene_items in tqdm(best_dates, desc='按日期处理'):
    try:
        day_stack = stackstac.stack(
            scene_items,
            assets=['B02','B03','B04','B08','B11'],
            bounds_latlon=BBOX,
            resolution=10,
            dtype='float64',
            rescale=False,
            epsg=EPSG
        )
        day_data = day_stack.compute()

        b04 = day_data.sel(band='B04').values / 10000.0
        b08 = day_data.sel(band='B08').values / 10000.0
        valid = (b04 > 0) & (b08 > 0) & np.isfinite(b04) & np.isfinite(b08)
        b04[~valid] = np.nan
        b08[~valid] = np.nan

        ndvi_day = (b08 - b04) / (b08 + b04 + 1e-8)
        ndvi_day[~valid] = np.nan

        # 对有全NaN的像素位置，best_t设为0
        has_valid = valid.any(axis=0)
        ndvi_max = np.where(has_valid, 
                            np.nanmax(np.where(valid, ndvi_day, -999), axis=0),
                            np.nan)
        best_t = np.where(has_valid,
                          np.nanargmax(np.where(valid, ndvi_day, -999), axis=0),
                          0).astype(int)

        rows, cols = np.indices(best_t.shape)

        b02 = day_data.sel(band='B02').values / 10000.0
        b03 = day_data.sel(band='B03').values / 10000.0
        b11 = day_data.sel(band='B11').values / 10000.0
        for b in [b02, b03, b04, b08, b11]:
            b[~valid] = np.nan

        daily_ndvi.append(ndvi_max)
        daily_bands.append([
            b02[best_t, rows, cols],
            b03[best_t, rows, cols],
            b04[best_t, rows, cols],
            b08[best_t, rows, cols],
            b11[best_t, rows, cols],
        ])

        if transform_ref is None:
            transform_ref = day_stack.transform
            height, width = ndvi_max.shape

        del day_data, day_stack
        gc.collect()
        valid_pct = np.isfinite(ndvi_max).mean() * 100
        print(f'  {date} 完成，有效像素: {valid_pct:.1f}%')

    except Exception as e:
        print(f'  {date} 跳过: {e}')

print(f'\n共处理 {len(daily_ndvi)} 个日期')

选取日期:
  2026-04-23: 8 景
  2026-04-21: 7 景
  2026-05-01: 6 景


按日期处理:  33%|███▎      | 1/3 [16:10<32:20, 970.32s/it]

  2026-04-23 完成，有效像素: 74.4%


按日期处理:  67%|██████▋   | 2/3 [27:15<13:11, 791.06s/it]

  2026-04-21 跳过: Error reading Window(col_off=9216, row_off=6144, width=1024, height=1024) from 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/55/H/ED/2026/04/21/S2C_MSIL2A_20260421T001111_N0512_R073_T55HED_20260421T040617.SAFE/GRANULE/L2A_T55HED_A008480_20260421T001226/IMG_DATA/R10m/T55HED_20260421T001111_B04_10m.tif?st=2026-05-17T07%3A45%3A29Z&se=2026-05-18T08%3A30%3A29Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-05-18T06%3A58%3A17Z&ske=2026-05-25T06%3A58%3A17Z&sks=b&skv=2025-07-05&sig=2/FIY4GF15UHboqNpk3lFRZPRFTR47VJiEQZzPCEHTo%3D': RasterioIOError('Read failed. See previous exception for details.')


按日期处理: 100%|██████████| 3/3 [27:17<00:00, 545.70s/it]

  2026-05-01 跳过: Error opening 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/55/H/ED/2026/05/01/S2C_MSIL2A_20260501T001111_N0512_R073_T55HED_20260501T041116.SAFE/GRANULE/L2A_T55HED_A008623_20260501T001118/IMG_DATA/R10m/T55HED_20260501T001111_B03_10m.tif?st=2026-05-17T07%3A45%3A29Z&se=2026-05-18T08%3A30%3A29Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-05-18T06%3A58%3A17Z&ske=2026-05-25T06%3A58%3A17Z&sks=b&skv=2025-07-05&sig=2/FIY4GF15UHboqNpk3lFRZPRFTR47VJiEQZzPCEHTo%3D': RasterioIOError('HTTP response code: 403')

共处理 1 个日期


## 6. 月合成（最大NDVI合成法）

对多景影像取月合成：每个像素选取该月内 NDVI 最高的无云观测值。
这是 C 因子研究中最常用的合成策略，能最大程度保留植被信号。

In [ ]:
# 提取各波段（时间 × 行 × 列）
B02 = data.sel(band='B02').values / 10000.0
B03 = data.sel(band='B03').values / 10000.0
B04 = data.sel(band='B04').values / 10000.0
B08 = data.sel(band='B08').values / 10000.0
B8A = data.sel(band='B8A').values / 10000.0
B11 = data.sel(band='B11').values / 10000.0

# 对无效像素设为 NaN
for band_arr in [B02, B03, B04, B08, B8A, B11]:
    band_arr[~valid_mask] = np.nan

# 计算每个时次的 NDVI
eps = 1e-8
ndvi_ts = (B08 - B04) / (B08 + B04 + eps)
ndvi_ts[~valid_mask] = np.nan

# 最大 NDVI 合成：每个像素选 NDVI 最高的时次
best_t = np.nanargmax(ndvi_ts, axis=0)  # 形状: (行, 列)
rows, cols = np.indices(best_t.shape)

# 用最优时次索引提取各波段合成值
B02_comp = B02[best_t, rows, cols]
B03_comp = B03[best_t, rows, cols]
B04_comp = B04[best_t, rows, cols]
B08_comp = B08[best_t, rows, cols]
B8A_comp = B8A[best_t, rows, cols]
B11_comp = B11[best_t, rows, cols]

print('月合成完成')
print(f'合成后形状: {B04_comp.shape}')
valid_pct = np.isfinite(B04_comp).mean() * 100
print(f'合成后有效像素比例: {valid_pct:.1f}%')

## 7. 计算合成后的遥感指数

In [ ]:
bands_comp = {
    'blue' : B02_comp,
    'green': B03_comp,
    'red'  : B04_comp,
    'nir'  : B08_comp,
    'swir' : B11_comp
}

indices = calculate_all_indices(bands_comp)

print(f'{"指数":<6} {"最小值":>8} {"最大值":>8} {"均值":>8} {"有效像素%":>10}')
print('-' * 46)
for name, arr in indices.items():
    valid = arr[np.isfinite(arr)]
    pct   = len(valid) / arr.size * 100
    print(f'{name:<6} {valid.min():>8.3f} {valid.max():>8.3f} {valid.mean():>8.3f} {pct:>10.1f}%')

## 8. 可视化月合成结果

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('NSW 测试区 Sentinel-2 月合成指数\n2026年4月（最大NDVI合成）', fontsize=14)

plots = [
    ('NDVI', indices['NDVI'], 'RdYlGn', -0.2, 0.8),
    ('NDWI', indices['NDWI'], 'Blues',  -0.4, 0.4),
    ('BSI',  indices['BSI'],  'YlOrBr', -0.3, 0.3),
    ('SAVI', indices['SAVI'], 'RdYlGn', -0.2, 0.8),
]

for ax, (name, arr, cmap, vmin, vmax) in zip(axes.flat, plots):
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(name, fontsize=12)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()

fig_dir  = ensure_dir('../results/figures')
fig_path = fig_dir / 'NSW_composite_indices_2026_04.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'图像已保存: {fig_path}')
plt.show()

## 9. 构建特征矩阵并保存

In [ ]:
processed_dir = ensure_dir('../data/processed')

# 获取空间参考
crs       = f'EPSG:{EPSG}'
transform = data.transform
height, width = B04_comp.shape

# 保存各指数 GeoTIFF
for name, arr in indices.items():
    out_path = processed_dir / f'NSW_test_{name}_composite_2026_04.tif'
    with rasterio.open(
        out_path, 'w',
        driver='GTiff',
        height=height, width=width,
        count=1,
        dtype='float32',
        crs=crs,
        transform=transform,
        compress='lzw',
        nodata=np.nan
    ) as dst:
        dst.write(arr.astype('float32'), 1)
    print(f'已保存: {out_path.name}')

# 保存特征矩阵为 numpy（行×列×通道）
feature_stack = np.stack([
    indices['NDVI'],
    indices['NDWI'],
    indices['BSI'],
    indices['SAVI'],
    B04_comp,   # Red
    B08_comp,   # NIR
    B11_comp    # SWIR
], axis=-1)  # 形状: (H, W, 7)

npy_path = processed_dir / 'NSW_test_features_2026_04.npy'
np.save(npy_path, feature_stack.astype('float32'))
print(f'\n特征矩阵已保存: {npy_path.name}')
print(f'特征矩阵形状  : {feature_stack.shape}  (H × W × 7通道)')
print(f'7通道依次为   : NDVI, NDWI, BSI, SAVI, B04, B08, B11')

## 10. 提交到 GitHub

In [ ]:
import subprocess

commands = [
    ['git', 'add', 'notebooks/02_feature_engineering.ipynb'],
    ['git', 'add', 'results/figures/NSW_composite_indices_2026_04.png'],
    ['git', 'commit', '-m', 'feat: notebook 02 - multi-scene mosaic, cloud mask, feature matrix'],
    ['git', 'push']
]

for cmd in commands:
    result = subprocess.run(cmd, cwd='..', capture_output=True, text=True)
    print(' '.join(cmd))
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)